# Phase 1: Khám phá và tạo feature

Notebook tạo dữ liệu sạch, Elo và form 5 trận gần nhất cho model.

In [68]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
MODEL_START = pd.Timestamp("2010-01-01")
ELO_WARMUP_START = pd.Timestamp("2000-01-01")
FORM_WINDOW = 5

In [69]:
df = pd.read_csv(DATA_URL)
df["date"] = pd.to_datetime(df["date"])
print("Dữ liệu gốc:", df.shape)
print("Thiếu dữ liệu:\n", df.isna().sum())
print("Dòng trùng:", df.duplicated().sum())
df.head()

Dữ liệu gốc: (49477, 9)
Thiếu dữ liệu:
 date           0
home_team      0
away_team      0
home_score    32
away_score    32
tournament     0
city           0
country        0
neutral        0
dtype: int64
Dòng trùng: 0


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [70]:
def get_outcome(row):
    if pd.isna(row["home_score"]) or pd.isna(row["away_score"]):
        return pd.NA
    if row["home_score"] > row["away_score"]:
        return "home_win"
    if row["home_score"] < row["away_score"]:
        return "away_win"
    return "draw"


df["outcome"] = df.apply(get_outcome, axis=1)
assert df.loc[df["home_score"].isna(), "outcome"].isna().all()
print(df["outcome"].value_counts(normalize=True))
print("\nTheo neutral:")
print(df.groupby("neutral")["outcome"].value_counts(normalize=True))

outcome
home_win    0.490080
away_win    0.282475
draw        0.227445
Name: proportion, dtype: float64

Theo neutral:
neutral  outcome 
False    home_win    0.507468
         away_win    0.263967
         draw        0.228564
True     home_win    0.441797
         away_win    0.333868
         draw        0.224335
Name: proportion, dtype: float64


In [71]:
model_scope = df[
    df["date"].ge(MODEL_START)
    & df["tournament"].ne("Friendly")
].copy()
upcoming_matches = model_scope[
    model_scope[["home_score", "away_score"]].isna().any(axis=1)
].copy()
df_recent = (
    model_scope.dropna(subset=["home_score", "away_score"])
    .sort_values("date")
    .reset_index(drop=True)
)
assert df_recent["outcome"].notna().all()
assert upcoming_matches["outcome"].isna().all()
print("Đã đá:", df_recent.shape, "| Chưa đá:", upcoming_matches.shape)

Đã đá: (10784, 10) | Chưa đá: (32, 10)


## Elo Rating

$$E_A = \frac{1}{1 + 10^{-(R_A-R_B)/400}}$$

$E_A$ là expected score; hòa được tính là 0,5. Elo được warm-up bằng các trận 2000–2009 trước khi tạo feature từ năm 2010.

In [72]:
def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** (-(elo_a - elo_b) / 400))


def update_elo(elo, expected, actual, k=20):
    return elo + k * (actual - expected)


OUTCOME_SCORE = {"home_win": 1.0, "draw": 0.5, "away_win": 0.0}
warmup_matches = df[
    df["date"].between(ELO_WARMUP_START, MODEL_START, inclusive="left")
    & df["tournament"].ne("Friendly")
].dropna(subset=["home_score", "away_score"]).sort_values("date")
all_teams = (
    set(warmup_matches["home_team"]) | set(warmup_matches["away_team"])
    | set(df_recent["home_team"]) | set(df_recent["away_team"])
    | set(upcoming_matches["home_team"]) | set(upcoming_matches["away_team"])
)
elo = {team: 1500.0 for team in all_teams}

def apply_elo_result(row):
    home_before, away_before = elo[row.home_team], elo[row.away_team]
    expected_home = expected_score(home_before, away_before)
    actual_home = OUTCOME_SCORE[row.outcome]
    elo[row.home_team] = update_elo(home_before, expected_home, actual_home)
    elo[row.away_team] = update_elo(away_before, 1 - expected_home, 1 - actual_home)

for row in warmup_matches.itertuples(index=False):
    apply_elo_result(row)

home_elo_before, away_elo_before = [], []
for row in df_recent.itertuples(index=False):
    home_elo_before.append(elo[row.home_team])
    away_elo_before.append(elo[row.away_team])
    apply_elo_result(row)

df_recent["home_elo_before"] = home_elo_before
df_recent["away_elo_before"] = away_elo_before
df_recent["elo_diff"] = df_recent["home_elo_before"] - df_recent["away_elo_before"]
upcoming_matches["home_elo_now"] = upcoming_matches["home_team"].map(elo)
upcoming_matches["away_elo_now"] = upcoming_matches["away_team"].map(elo)
upcoming_matches["elo_diff"] = upcoming_matches["home_elo_now"] - upcoming_matches["away_elo_now"]
assert upcoming_matches[["home_elo_now", "away_elo_now", "elo_diff"]].notna().all().all()
print("Warm-up:", len(warmup_matches), "trận")

Warm-up: 6151 trận


## Form 5 trận gần nhất

Feature gồm điểm trung bình, bàn thắng và bàn thua. Mỗi trận chỉ nhìn các ngày trước đó; các trận cùng ngày không nhìn thấy kết quả của nhau.

In [73]:
df_recent["match_id"] = np.arange(len(df_recent))
home_history = df_recent[["match_id", "date", "home_team", "home_score", "away_score", "outcome"]].rename(
    columns={"home_team": "team", "home_score": "gf", "away_score": "ga"}
)
home_history["points"] = home_history["outcome"].map({"home_win": 3, "draw": 1, "away_win": 0})
home_history["side"] = "home"
away_history = df_recent[["match_id", "date", "away_team", "away_score", "home_score", "outcome"]].rename(
    columns={"away_team": "team", "away_score": "gf", "home_score": "ga"}
)
away_history["points"] = away_history["outcome"].map({"home_win": 0, "draw": 1, "away_win": 3})
away_history["side"] = "away"
team_history = (
    pd.concat([home_history, away_history], ignore_index=True)
    .sort_values(["team", "date", "match_id"])
    .reset_index(drop=True)
)
for column in ["points_form", "gf_form", "ga_form"]:
    team_history[column] = np.nan

for _, team_matches in team_history.groupby("team", sort=False):
    history = []
    for _, same_day in team_matches.groupby("date", sort=True):
        prior = history[-FORM_WINDOW:]
        if prior:
            team_history.loc[same_day.index, "points_form"] = np.mean([item[0] for item in prior])
            team_history.loc[same_day.index, "gf_form"] = np.mean([item[1] for item in prior])
            team_history.loc[same_day.index, "ga_form"] = np.mean([item[2] for item in prior])
        history.extend(zip(same_day["points"], same_day["gf"], same_day["ga"]))

for side in ["home", "away"]:
    lookup = team_history[team_history["side"].eq(side)].set_index("match_id")
    for statistic in ["points_form", "gf_form", "ga_form"]:
        df_recent[f"{side}_{statistic}"] = df_recent["match_id"].map(lookup[statistic])

current_form = {}
for team, team_matches in team_history.groupby("team"):
    recent = team_matches.sort_values(["date", "match_id"]).tail(FORM_WINDOW)
    current_form[team] = {
        "points_form": recent["points"].mean(),
        "gf_form": recent["gf"].mean(),
        "ga_form": recent["ga"].mean(),
    }
for side in ["home", "away"]:
    for statistic in ["points_form", "gf_form", "ga_form"]:
        upcoming_matches[f"{side}_{statistic}"] = upcoming_matches[f"{side}_team"].map(
            lambda team: current_form.get(team, {}).get(statistic, np.nan)
        )
df_recent = df_recent.drop(columns="match_id")
FORM_FEATURES = [
    "home_points_form", "away_points_form",
    "home_gf_form", "away_gf_form",
    "home_ga_form", "away_ga_form",
]
print(df_recent[FORM_FEATURES].isna().sum())
assert upcoming_matches[FORM_FEATURES].notna().all().all()

home_points_form    142
away_points_form    153
home_gf_form        142
away_gf_form        153
home_ga_form        142
away_ga_form        153
dtype: int64


In [74]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
df_recent.to_csv(output_dir / "matches_features.csv", index=False)
upcoming_matches.to_csv(output_dir / "upcoming_matches.csv", index=False)
print("Đã lưu:", df_recent.shape, upcoming_matches.shape)

Đã lưu: (10784, 19) (32, 19)
